# Advanced AI Excel Pipeline

This notebook demonstrates how an AI Data Scientist handles complex Excel files with formulas.

We will:
1. Create a dummy Excel file with a formula (`=A2*B2`).
2. Show how standard `pandas` fails to see the formula.
3. Use `openpyxl` to extract the exact formula string.
4. Ask the AI (`gpt-4.1-mini`) to translate that Excel formula into Python (`pandas`) code!

In [ ]:
!pip install openpyxl pandas python-dotenv dspy-ai

### Step 1: Create a Dummy Excel File with Formulas
Let's create a `.xlsx` file that tracks Sales and Price, with a formula for Total Revenue.

In [ ]:
import pandas as pd
import openpyxl
from openpyxl import Workbook
import os

# Create an Excel file with a formula
file_path = "Sales_Data_With_Formulas.xlsx"

wb = Workbook()
ws = wb.active
ws.title = "Sales"

# Add Headers
ws['A1'] = "Items Sold"
ws['B1'] = "Price Per Item"
ws['C1'] = "Total Revenue"

# Add Data
ws['A2'] = 10
ws['B2'] = 25.50

# ADD THE EXCEL FORMULA!
ws['C2'] = "=A2*B2"  # 10 * 25.50 = 255

# Save the file
wb.save(file_path)
print(f"Successfully created {file_path} with a formula in cell C2!")

### Step 2: Extracting the Raw Formula using `openpyxl`
If we use standard `pandas`, we just get empty values (because the formula hasn't been cached by Microsoft Excel yet!). Instead, we will use `openpyxl` to read the raw formula strings so we can give them to the AI.

In [ ]:
# Load the workbook, explicitly telling openpyxl NOT to compute data only
wb_formulas = openpyxl.load_workbook(file_path, data_only=False)
sheet = wb_formulas.active

formula_found = sheet['C2'].value

print("RAW EXCEL VALUE EXTRACTED FROM C2:")
print(formula_found)

# We will bundle this into our context for the AI
excel_metadata = {
    "columns": ["Items Sold", "Price Per Item", "Total Revenue"],
    "formulas_detected": {
        "Total Revenue": formula_found
    }
}

import json
print("\nThis is what we will send to the AI:")
print(json.dumps(excel_metadata, indent=2))

### Step 3: Auto-Translate Excel Logic to Python!
Now we give this JSON context to our AI Data Scientist and ask it to write pure Python pandas code to permanently replace the fragile Excel formula.

In [ ]:
import dspy
from dotenv import load_dotenv
import os
import sys

load_dotenv()

# We will temporarily append the backend path so we can use our model_registry
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils.model_registry import get_model_object

# Configure DSPy to use our gpt-4.1-mini
dspy.settings.configure(lm=get_model_object("gpt-4.1-mini"))

prompt = f"""
You are an AI Data Scientist. Your goal is to translate raw Excel formulas into Python Pandas code.

Here is the schema and the raw formulas we extracted from the user's uploaded Excel file:
{json.dumps(excel_metadata, indent=2)}

User Query: Can you write a python script that accurately reproduces the 'Total Revenue' formula so I don't need the Excel file anymore?

Rules:
1. Assume the data is loaded in a DataFrame called `df`.
2. Write ONLY standard Python code. Do not wrap it in markdown block quotes.
3. Simply overwrite the existing column with the new calculated vectorized pandas values.
"""

print("Requesting translation from OpenAI...")

try:
    lm = get_model_object("gpt-4.1-mini")
    response = lm(prompt)
    code = response[0] if isinstance(response, list) else response
    code = code.replace("```python", "").replace("```", "").strip()
    
    print("\n--- AI TRANSLATED PYTHON CODE ---")
    print(code)
except Exception as e:
    print(f"Error: {e}")

### Step 4: Execute the AI's Translated Code
Now we load the flat data in Pandas, and run the AI's translated Python script to dynamically calculate the missing Total Revenue column in pure Python!

In [ ]:
# 1. Load the raw data (Pandas gives us a NaN or None for the uncalculated formula)
# Because we just created it and Excel didn't "cache" the calculation, pandas sees nothing.
df = pd.read_excel(file_path)
print("--- BEFORE RUNNING AI CODE ---")
print(df)

print("\n--- EXECUTING AI PYTHON SCRIPT LOCALLY ---")
# 2. We run the AI's translated Python code
local_env = {"df": df, "pd": pd}
exec(code, {}, local_env)
df_final = local_env["df"]

print("\n--- FINAL DATAFRAME RESULTS ---")
print(df_final)